# Synthetic dataset exploration

Sprint 2: inspect provenance, balance, grouping, and numeric distributions. This is
not a model evaluation. All examples are invented locally; positive labels mean
credential-shaped, not valid credentials. Negative labels apply to the authored
context. No sample values or source snippets are displayed.

First run `python -m ml.scripts.build_dataset` from the repository root. Open this
notebook using a Python 3.11+ Jupyter kernel with the core package installed.
The analysis uses only the standard library and the local core package. Jupyter
is optional and is not added to the scanner's runtime dependencies.


In [ ]:
import csv
import hashlib
import json
import os
import sys
from collections import Counter, defaultdict
from pathlib import Path
from statistics import mean, median

# Support starting Jupyter from either the repository or notebook directory.
root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "ml/scripts/build_dataset.py").is_file()
)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from secretsense.scanner.entropy import shannon_entropy

from ml.scripts.build_dataset import validate_rows

data_dir = Path(os.environ.get("SECRETSENSE_DATA_DIR", root / "ml/data"))
data = (data_dir / "dataset.csv").read_bytes()
manifest = json.loads((data_dir / "manifest.json").read_text(encoding="utf-8"))
assert hashlib.sha256(data).hexdigest() == manifest["dataset_sha256"], "Dataset checksum mismatch."
with (data_dir / "dataset.csv").open(encoding="utf-8", newline="") as handle:
    rows = list(csv.DictReader(handle))
validate_rows(rows)
assert len(rows) == manifest["rows"], "Manifest row count mismatch."
print(f"Verified {len(rows)} synthetic rows; generator version {manifest['generator_version']}.")
print(manifest["provenance"])
print(manifest["label_policy"])

## Balance and source isolation

The default dataset has 3,000 examples per label. A 60/20/20 split targets
3,600 training, 1,200 validation, and 1,200 test examples. The same source group
or generator family must never occur in multiple splits. Shared rendering
wrappers do not define independent groups.


In [ ]:
counts = Counter((row["split"], row["label"]) for row in rows)
print("split        label  rows")
for (split, label), count in sorted(counts.items()):
    print(f"{split:12} {label:5} {count:5}")
for column in ("source_group", "template_family"):
    memberships = defaultdict(set)
    for row in rows:
        memberships[row[column]].add(row["split"])
    assert all(len(splits) == 1 for splits in memberships.values()), "Group leakage."
    print(f"{column}: {len(memberships)} isolated groups")
print(f"Unique texts: {len({row['sample_id'] for row in rows})}")
print(f"Unique values: {len({row['value'] for row in rows})}")

## Family coverage

Entire families are held out, so each split contains different categories. This
is a deliberate generalization challenge, but only twenty authored families are
available. Evaluation on this dataset alone cannot establish performance on
unseen real repositories. The metadata below is for auditing, never model input.


In [ ]:
families = Counter((row["split"], row["label"], row["template_family"]) for row in rows)
print("split        label family                        rows")
for (split, label, family), count in sorted(families.items()):
    print(f"{split:12} {label:5} {family:29} {count:5}")

## Length and entropy exploration

Length and Shannon entropy are descriptive statistics, not predicted
probabilities. Repeated prefixes and distinct families may make the synthetic
task artificially easy. Check both labels and splits before choosing features.


In [ ]:
groups = defaultdict(list)
for row in rows:
    groups[row["split"], row["label"]].append(row["value"])
print("split        label  min_len median_len max_len mean_entropy")
for (split, label), values in sorted(groups.items()):
    lengths = [len(value) for value in values]
    entropy = mean(shannon_entropy(value) for value in values)
    print(
        f"{split:12} {label:5} {min(lengths):7} {median(lengths):10.1f} "
        f"{max(lengths):7} {entropy:12.3f}"
    )

print("\nValue-length histogram (one # represents approximately 50 rows)")
bounds = [(0, 31), (32, 63), (64, 127), (128, 255), (256, 1023)]
for label in ("0", "1"):
    for low, high in bounds:
        count = sum(row["label"] == label and low <= len(row["value"]) <= high for row in rows)
        print(f"label {label}, {low:3}-{high:4}: {count:4} {'#' * round(count / 50)}")

## Findings and next steps

- Labels are balanced by construction, not representative of secret prevalence.
- Provider tokens are random shapes, JWT signatures are random bytes, and private
  key bodies are deliberately invalid. No credentials are checked remotely.
- Negative templates are reviewed by local inspection, with rationale and
  limitations in `ml/data/negative_templates.json`. This is not independent human
  adjudication. No public test fixtures were imported.
- Near-duplicate generator patterns remain inside families. Exact value/text
  deduplication and group isolation reduce leakage without removing synthetic bias.
- Context and value may be used in future feature extraction. Exclude labels,
  IDs, categories, provenance, review status, and split metadata from features.
- Sprint 3 should assess candidate extraction coverage before training a local
  baseline, tune on validation only, and reserve the test split for final evaluation.
  Real-repository evaluation and harder independently reviewed negatives remain
  necessary before making real-world performance claims.
